# Quant Lab: Backtest Simulation & Constrained Optimization (QL-3)

This notebook demonstrates the offline research pipeline for **Robot Trade Quant Lab**:
- Generates reproducible synthetic OHLCV market data with cryptographic SHA-256 manifests.
- Executes the verified `synthetic-ema-v1` reference strategy under Spot long-only rules.
- Reconciles realized PnL and trade statistics against QL-2 FIFO analytics (80-digit precision).
- Performs chronological validation (Train/Validation/Test) and walk-forward verification.
- Runs constrained parameter optimization over approved bounds (`ParameterBounds`).
- Generates pre-flight Risk Simulation Previews and immutable `OptimizationRun` contract records.

In [ ]:
import pprint
from decimal import Decimal

from robot_quant import (
    BacktestConfig,
    ConstrainedOptimizer,
    ParameterBounds,
    RiskProfile,
    Scope,
    StrategyDefinition,
    SyntheticEmaParameters,
    SyntheticEmaStrategy,
    chronological_split,
    generate_risk_preview,
    generate_synthetic_ohlcv,
    run_backtest,
)

print("Quant Lab QL-3 environment loaded successfully.")

## 1. Deterministic Market Data Generation

In [ ]:
# Generate 1 year (8,760 hours) of deterministic synthetic BTC/USDT candles
candles = generate_synthetic_ohlcv(
    symbol="BTCUSDT",
    bars_count=8760,
    seed=42,
    initial_price=Decimal("42000.00"),
)
print(f"Generated {len(candles)} candles. Close: {candles[0].close} -> {candles[-1].close}")

## 2. Baseline Strategy Backtest & FIFO Reconciliation

In [ ]:
config = BacktestConfig(
    initial_capital=Decimal("10000.00"),
    requested_risk_percent=Decimal("25.0"),  # 25% of balance per position
    max_order_notional=Decimal("2500.00"),
    fee_bps=Decimal("10"),
    slippage_bps=Decimal("5"),
)

baseline_strategy = SyntheticEmaStrategy(
    SyntheticEmaParameters(ema_fast=10, ema_slow=30, atr_period=14, atr_multiplier=Decimal("2.0"))
)

baseline_result = run_backtest(candles, strategy=baseline_strategy, config=config)
print(f"Trades count: {len(baseline_result.signals)}")
print(f"Final cash: {baseline_result.final_cash}")
print(f"Net Profit: {baseline_result.metrics.net_profit} {baseline_result.metrics.currency}")
print(f"Win Rate: {baseline_result.metrics.win_rate}%")
print(f"Profit Factor: {baseline_result.metrics.profit_factor}")

## 3. Chronological Train / Validation / Test Splitting

In [ ]:
split = chronological_split(candles, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2)
print(f"Train bars: {len(split.train_candles)} (until ts={split.train_end})")
print(f"Validation bars: {len(split.val_candles)} (until ts={split.validation_end})")
print(f"Test bars: {len(split.test_candles)} (until ts={split.test_end})")

## 4. Constrained Parameter Optimization

In [ ]:
scope = Scope(
    owner_id="research_user",
    bot_id="bot_notebook",
    account_id="paper",
    broker="binance-global",
    currency="USDT",
)

profile = RiskProfile(
    risk_profile_id="risk_nb_01",
    version=1,
    scope=scope,
    effective_at=0,
    capital_basis="COST_BASIS_NOT_MARK_TO_MARKET",
    initial_capital=Decimal("10000.00"),
    balance=Decimal("10000.00"),
    max_risk_percent=Decimal("100.0"),
    requested_risk_percent=Decimal("25.0"),
    max_order_notional=Decimal("2500.00"),
    max_daily_notional=Decimal("25000.00"),
)

strat_def = StrategyDefinition(
    strategy_id="strat_ema_v1",
    version=1,
    template_id="synthetic-ema-v1",
    source_sha256="f" * 64,
    alert_source="alert_calls",
    parameters=(
        ParameterBounds(
            name="ema_fast",
            unit="bars",
            minimum=Decimal("6"),
            maximum=Decimal("14"),
            step=Decimal("2"),
            default=Decimal("10"),
            optimizable=True,
        ),
        ParameterBounds(
            name="ema_slow",
            unit="bars",
            minimum=Decimal("25"),
            maximum=Decimal("40"),
            step=Decimal("5"),
            default=Decimal("30"),
            optimizable=True,
        ),
        ParameterBounds(
            name="atr_period",
            unit="bars",
            minimum=Decimal("14"),
            maximum=Decimal("14"),
            step=Decimal("1"),
            default=Decimal("14"),
            locked=True,
            optimizable=False,
        ),
        ParameterBounds(
            name="atr_multiplier",
            unit="multiplier",
            minimum=Decimal("2.0"),
            maximum=Decimal("2.0"),
            step=Decimal("0.5"),
            default=Decimal("2.0"),
            locked=True,
            optimizable=False,
        ),
    ),
)

optimizer = ConstrainedOptimizer(
    strategy_def=strat_def,
    risk_profile=profile,
    candles=candles,
    objective="net_return",
    search_budget=30,
)

report = optimizer.optimize()
print(f"Total candidates evaluated: {report.total_evaluated}")
print(f"Rejection statistics: {report.rejection_stats}")
if report.best_candidate:
    print(f"Best candidate params: {report.best_candidate.params}")
    print(f"Validation Return: {report.best_candidate.val_return:.2f}")
    print(f"Out-of-sample Test Return: {report.best_candidate.test_return:.2f}")

## 5. Risk Simulation Preview

In [ ]:
preview = generate_risk_preview(profile, reference_price=Decimal("50000.00"))
pprint.pprint(preview.to_dict())